## Import libs

In [1]:
!pip -q install torch sentencepiece records babel tqdm tabulate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.5 MB/s eta 0:00:00


In [39]:
import json
import math
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path


### Download data from github

In [ ]:
!git clone https://github.com/salesforce/WikiSQL

fatal: destination path 'WikiSQL' already exists and is not an empty directory.


In [24]:
%cd WikiSQL
!ls


/content/WikiSQL
annotate.py		  data		LICENSE		  version.txt
CODEOWNERS		  data.tar.bz2	README.md
collection		  evaluate.py	requirements.txt
CONTRIBUTING-ARCHIVED.md  lib		test


In [25]:
!tar xvjf data.tar.bz2
%cd ..

data/
data/train.jsonl
data/test.tables.jsonl
data/test.db
data/dev.tables.jsonl
data/dev.db
data/test.jsonl
data/train.tables.jsonl
data/train.db
data/dev.jsonl
/content


### Load & preprocess

In [ ]:
DATA_DIR = Path("WikiSQL/data")
AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<"]
MAX_COLS = 64  # column tokens <c0> ... <c63>


def load_split(split):
    """Return (examples, tables) for 'train', 'dev' or 'test'."""
    tables = {}

    with open(DATA_DIR / f"{split}.tables.jsonl", encoding="utf-8") as f:
        for line in f:
            t = json.loads(line)
            tables[t["id"]] = t

    with open(DATA_DIR / f"{split}.jsonl", encoding="utf-8") as f:
        examples = [json.loads(line) for line in f]

    return examples, tables


def encode_source(question, header):
    """question <sep> <c0> col name <c1> col name ..."""
    cols = " ".join(
        f"<c{i}> {name}" for i, name in enumerate(header)
    )
    return f"{question.strip()} <sep> {cols}".lower()


def encode_target(sql):
    """{'sel','agg','conds'} -> 'select count <c3> where <c1> = kim manners'"""
    out = ["select"]

    if sql["agg"]:
        out.append(AGG_OPS[sql["agg"]].lower())

    out.append(f"<c{sql['sel']}>")

    for i, (col, op, val) in enumerate(sql["conds"]):
        out += [
            "where" if i == 0 else "and",
            f"<c{col}>",
            COND_OPS[op],
            str(val)
        ]

    return " ".join(out).lower()


def build_pairs(split):
    examples, tables = load_split(split)
    pairs = []

    for ex in examples:
        header = tables[ex["table_id"]]["header"]

        pairs.append({
            "table_id": ex["table_id"],
            "src": encode_source(ex["question"], header),
            "tgt": encode_target(ex["sql"]),
        })

    return pairs

In [35]:
for split in ["train", "dev", "test"]:
        pairs = build_pairs(split)

        with open(
            f"{split}_pairs.jsonl", "w", encoding="utf-8"
        ) as f:
            for p in pairs:
                f.write(json.dumps(p, ensure_ascii=False) + "\n")

        print(f"{split}: {len(pairs)} pairs")
        print(pairs[0]["src"])
        print(pairs[0]["tgt"])

train: 56355 pairs
tell me what the notes are for south australia <sep> <c0> state/territory <c1> text/background colour <c2> format <c3> current slogan <c4> current series <c5> notes
select <c5> where <c3> = south australia
dev: 8421 pairs
what position does the player who played for butler cc (ks) play? <sep> <c0> player <c1> no. <c2> nationality <c3> position <c4> years in toronto <c5> school/club team
select <c3> where <c5> = butler cc (ks)
test: 15878 pairs
what is terrence ross' nationality <sep> <c0> player <c1> no. <c2> nationality <c3> position <c4> years in toronto <c5> school/club team
select <c2> where <c0> = terrence ross


### Tokenizer

In [ ]:
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPECIAL = ["<sep>"] + [f"<c{i}>" for i in range(MAX_COLS)]

def read_pairs(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def train_tokenizer(train_path="train_pairs.jsonl", vocab_size=8000):
    pairs = read_pairs(train_path)

    with open("spm_corpus.txt", "w", encoding="utf-8") as f:
        for p in pairs:  # ONE shared vocabulary for
            f.write(p["src"] + "\n")  # source and target, as in
            f.write(p["tgt"] + "\n")  # the original Transformer

    spm.SentencePieceTrainer.train(
        input="spm_corpus.txt",
        model_prefix="sql_sp",
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=1.0,
        user_defined_symbols=SPECIAL,  # never split <sep>, <c0>, ...
        pad_id=PAD_ID,
        unk_id=UNK_ID,
        bos_id=BOS_ID,
        eos_id=EOS_ID,
    )

    return spm.SentencePieceProcessor(model_file="sql_sp.model")
    

In [32]:
sp = train_tokenizer()
p = read_pairs("dev_pairs.jsonl")[0]

print(sp.encode(p["src"], out_type=str))
print(sp.encode(p["tgt"], out_type=str))
print(sp.decode(sp.encode(p["tgt"])) == p["tgt"])

['▁what', '▁position', '▁does', '▁the', '▁player', '▁who', '▁played', '▁for', '▁but', 'ler', '▁cc', '▁(', 'ks', ')', '▁play', '?', '▁', '<sep>', '▁', '<c0>', '▁player', '▁', '<c1>', '▁no', '.', '▁', '<c2>', '▁nationality', '▁', '<c3>', '▁position', '▁', '<c4>', '▁years', '▁in', '▁toronto', '▁', '<c5>', '▁school', '/', 'club', '▁team']
['▁select', '▁', '<c3>', '▁where', '▁', '<c5>', '▁=', '▁but', 'ler', '▁cc', '▁(', 'ks', ')']
True


### Dataloaders
- Tensor & Batches

In [38]:
class SQLDataset(Dataset):
    """
    Each item: (source ids, target ids). Target = <s> ... </s>.
    train=True drops over-long pairs.
    train=False keeps EVERY row in file order (needed for evaluation).
    """

    def __init__(self, path, sp, train=True, max_src=160, max_tgt=64):
        self.items, skipped = [], 0

        for p in read_pairs(path):
            src = sp.encode(p["src"]) + [EOS_ID]
            tgt = [BOS_ID] + sp.encode(p["tgt"]) + [EOS_ID]

            if train and (len(src) > max_src or len(tgt) > max_tgt):
                skipped += 1
                continue

            self.items.append((src, tgt))

        print(f"{path}: kept {len(self.items)}, skipped {skipped}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    srcs, tgts = zip(*batch)

    S, T = max(map(len, srcs)), max(map(len, tgts))

    src = torch.full(
        (len(batch), S),
        PAD_ID,
        dtype=torch.long
    )

    tgt = torch.full(
        (len(batch), T),
        PAD_ID,
        dtype=torch.long
    )

    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)

    return src, tgt  # (B, S), (B, T)


def make_loader(path, sp, train, batch_size=64):
    return DataLoader(
        SQLDataset(path, sp, train=train),
        batch_size=batch_size,
        shuffle=train,
        collate_fn=collate
    )

### Embeddings & Positional encoding

In [40]:
class TokenEmbedding(nn.Module):
    """
    Section 3.4: embeddings are multiplied by sqrt(d_model).
    One instance is shared by the encoder, the decoder and
    (optionally) the final output projection.
    """

    def __init__(self, vocab_size, d_model, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=pad_id
        )
        self.scale = math.sqrt(d_model)

    def forward(self, ids):  # (B, L) -> (B, L, d_model)
        return self.emb(ids) * self.scale


class PositionalEncoding(nn.Module):
    """
    Section 3.5: fixed sinusoidal encodings.
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pos = torch.arange(max_len).unsqueeze(1)  # (L, 1)

        div = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )  # (d/2,)

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )  # (1, L, d)

    def forward(self, x):  # x: (B, L, d_model)
        return self.dropout(
            x + self.pe[:, :x.size(1)]
        )


class InputLayer(nn.Module):
    """
    token ids -> scaled embedding + positional encoding -> dropout
    """

    def __init__(
        self,
        token_emb,
        d_model,
        max_len=512,
        dropout=0.1
    ):
        super().__init__()

        self.tok = token_emb
        self.pos = PositionalEncoding(
            d_model,
            max_len,
            dropout
        )

    def forward(self, ids):
        return self.pos(self.tok(ids))

### Check eveything

In [41]:
sp = spm.SentencePieceProcessor(model_file="sql_sp.model")

train_dl = make_loader("train_pairs.jsonl", sp, train=True)
dev_dl = make_loader("dev_pairs.jsonl", sp, train=False)


src, tgt = next(iter(train_dl))

print("src", tuple(src.shape), "tgt", tuple(tgt.shape))


d_model = 256

shared = TokenEmbedding(
    sp.get_piece_size(),
    d_model,
    PAD_ID
)

enc_in = InputLayer(shared, d_model)  # encoder input
dec_in = InputLayer(shared, d_model)  # decoder input (same weights)


x = enc_in(src)  # (B, S, 256) -> goes into YOUR encoder
y = dec_in(tgt[:, :-1])  # (B, T-1, 256) -> goes into YOUR decoder

print(
    "encoder input",
    tuple(x.shape),
    "decoder input",
    tuple(y.shape)
)

train_pairs.jsonl: kept 56336, skipped 19
dev_pairs.jsonl: kept 8421, skipped 0
src (64, 96) tgt (64, 30)
encoder input (64, 96, 256) decoder input (64, 29, 256)


### Achitecture